# Module 3: Measuring and Trusting Jev

*Getting Started with Jev - Data Trainers LLC - Axel Sirota*

**Time alone:** about 100 minutes, plus about 30 for the optional Level 3 labs: about 40 min
reading, about 15 min waiting on cells (a 1.6 GB model download and about 1,000 Jev calls),
about 45 min in the labs. This is the heaviest module of the course.

## The story

Our triage layer for the personal assistant app now gives every message a team, and Module 2
taught it to describe each team by its own 15 intents. Every answer comes with a probability,
and our code already acts on those numbers: route this one, send that one to a person.

But we never checked the numbers. Is an answer at 0.95 right 95 times in 100? What happens to a
message no team handles? And the review threshold we used was a guess. In this module we measure
Jev on labeled data, compare it with a free zero-shot model from Hugging Face, catch the messages
that fit no team, and choose the review threshold from data.

## What you will be able to do

- Measure accuracy and calibration on labeled data, and read a reliability plot
- Catch messages no team handles by pairing a Choice with a companion Noul
- Choose a human-review threshold from a stated target on one labeled set, and check it on another
- Compare Jev with a zero-shot classifier on accuracy, calibration, speed and cost

## Prerequisites

- Modules 1 and 2 (Choice, Noul, probabilities; criteria as data; judgments in Jev, decisions in code).
  The recap below covers what this module uses
- Basic classification metrics: accuracy, a labeled test set
- A Jev API key: create one in the TypeSafe console at https://console.typesafe.ai/keys (in a
  live class the instructor hands one out). No OpenAI key in this module
- A GPU runtime, recommended: in Colab, Runtime > Change runtime type > T4 GPU, before you run
  anything. The zero-shot baseline runs on it; Jev does not need it

## How this notebook works

Three Parts, each in four beats:

| Beat | What happens |
|---|---|
| 1. The problem | what goes wrong, with a real example |
| 2. The solution | the idea that fixes it |
| 3. Demo | you run it: first the problem, then the solution |
| 4. Lab | you build it, in three levels: Guided, Core, Challenge (optional) |

## From earlier modules

**Figure: What Modules 1 and 2 built, and what this module measures.**

<!-- DIAGRAM: recap -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/recap.png" width="740" alt="STATE: a dict of facts Jev reads, here the message. QUESTIONS in one request: a Choice picks an option, a Score picks a level, a Noul asks whether a statement is true; criteria are a sentence, a list or data such as TEAM_INTENTS. JEV ANSWERS, typed: .choice, .probabilities and .confidence; .score and .confidence; .noul, the probability the statement is true. YOUR CODE decides, from Modules 1 and 2: Noul bands, below 0.3 no, 0.7 or more yes, between a person looks; REVIEW_BELOW 0.5, a guess with no data yet. MODULE 3 measures: is 0.95 right 95 in 100, what about a message no team handles, and the review line from data">

- A **state** is a plain dict of facts (here `{"message": ...}`); each **question** asks one thing about it, and all of them go in one `jev.system_one(state=..., questions=..., model=...)` call
- **Criteria** say what each option means: a sentence, a list, or data. Module 2 described each team by its 15 intents, `TEAM_INTENTS`, and that question (`team_question_data` here) got nearly every team right
- Jev judges, your code decides. This module checks the numbers that code acted on: the probabilities, the 0.3 / 0.7 Noul bands, and Module 2's guessed `REVIEW_BELOW` of 0.5


# Setup

About 10 minutes, most of it the download. Run the next five cells in order: install, key,
config, data, and the zero-shot model. The key
cell asks for your Jev key once; it stays in this runtime only and is never printed. The last
cell downloads a 1.6 GB model and says whether you have a GPU: wait for its "loaded" line before
you go on. No GPU? Switch the runtime to T4 now (the runtime restarts: run Setup again), or
stay on CPU and expect Labs 1 and 3 to wait a few minutes for the zero-shot model.

The data cell draws two labeled sets. **DEV** is for looking and choosing: we read it, plot it,
and pick thresholds on it. **TEST** is for one honest check at the end, on messages nothing was
chosen on.

**Figure: Two labeled sets - DEV is for looking and choosing, TEST for one check of what DEV chose.**

<!-- DIAGRAM: setup-dev-test-split -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/setup-dev-test-split.png" width="750" alt="CLINC150&#x27;s test split, 4,500 in scope and 1,000 out of scope, splits into DEV (seed 42: 200 in scope, 20 per team, plus 100 out of scope) for plots, sweeps and picking the threshold, and TEST (seed 7: 100 in scope, 10 per team, plus 50 out of scope, none of them in DEV) to check DEV&#x27;s pick once and never choose on it">

In [ ]:
%pip install -q typesafe-sdk==0.7.5 datasets==5.1.0

In [ ]:
# ===========================================================================
# KEYS: the Jev key, typed once, never printed
# ===========================================================================
# Why: keys never live in the notebook. If your environment already has it, we use it.
import os
from getpass import getpass

if not os.environ.get("TYPESAFE_API_KEY"):
    os.environ["TYPESAFE_API_KEY"] = getpass("TYPESAFE_API_KEY: ")
if not os.environ["TYPESAFE_API_KEY"].strip():
    raise ValueError("TYPESAFE_API_KEY is empty. Paste your key when asked.")
print("The Jev key is set for this runtime.")

In [ ]:
# ===========================================================================
# CONFIG: client, pinned model, price, one probe call
# ===========================================================================
# Why: today we choose thresholds from data. A threshold holds only for the model version it
# was measured on, so the model id is pinned.
# This notebook sends about 1,000 Jev calls. If several people share one key (a class), a 429
# (rate limited) is possible: the client retries a rate-limited call up to 6 times, waiting
# longer each time. If a cell stalls, let it
# finish; if it fails, run it again.
import statistics
import time

from typesafe_sdk import Choice, Noul, RetryPolicy, TypeSafeClient

JEV_MODEL = "jev-1.14.0"            # pinned; "jev-latest" moves with each release
JEV_PRICE_IN = 0.042 / 1_000_000    # USD per input token; Jev output is free

jev = TypeSafeClient(retry=RetryPolicy(max_retries=6))
probe = jev.system_one(state={"message": "hello there"}, model=JEV_MODEL,
                       questions={"greeting": Noul(instructions="Is `message` a greeting?")})
assert probe.model == JEV_MODEL, f"expected {JEV_MODEL}, got {probe.model}"
print(f"Jev answered with {probe.model}: greeting = {probe.answers['greeting'].noul:.2f}")

In [ ]:
# ===========================================================================
# DATA: labeled CLINC150 messages, in scope and out of scope, split in two
# ===========================================================================
# Why: to measure Jev we need answers we already know. CLINC150's test split has 4,500
# in-scope messages (each with a team) and 1,000 out-of-scope ones that fit none of its 150
# intents. We draw two separate sets: DEV to look at and choose thresholds on, TEST to check
# them once at the end. Each row is (message, team); out-of-scope rows have the team "oos".
import json
import random
import urllib.request

from datasets import load_dataset

DOMAINS_URL = "https://raw.githubusercontent.com/clinc/oos-eval/master/data/domains.json"
TEAM_INTENTS = json.load(urllib.request.urlopen(DOMAINS_URL))     # team -> its 15 intents
intent_to_team = {intent: team for team, intents in TEAM_INTENTS.items() for intent in intents}

clinc = load_dataset("clinc/clinc_oos", "plus", split="test")
intent_names = clinc.features["intent"].names
messages = [(row["text"], intent_to_team.get(intent_names[row["intent"]], "oos")) for row in clinc]
in_scope = [m for m in messages if m[1] != "oos"]
out_of_scope = [m for m in messages if m[1] == "oos"]

TEAMS = {   # Module 1's short text descriptions, kept for Part 1
    "banking": "accounts, balances, transfers, bills, direct deposit",
    "credit_cards": "credit cards: limits, rewards, lost or damaged cards, APR",
    "kitchen_and_dining": "recipes, cooking, nutrition, restaurants",
    "home": "smart home, shopping and todo lists, reminders, calendar",
    "auto_and_commute": "car maintenance, gas, traffic, rides, directions",
    "travel": "flights, hotels, luggage, visas, travel alerts, translation",
    "utility": "time, date, weather, timers, calculator, definitions",
    "work": "payroll, PTO, taxes, 401k, meetings",
    "small_talk": "greetings, jokes, chit-chat with the assistant",
    "meta": "settings of the assistant itself: name, language, voice, volume",
}

rng = random.Random(42)
dev_in = []
for team in TEAMS:
    dev_in += rng.sample([m for m in in_scope if m[1] == team], 20)
dev_oos = rng.sample(out_of_scope, 100)
dev = dev_in + dev_oos

rng = random.Random(7)
test = []
for team in TEAMS:
    test += rng.sample([m for m in in_scope if m[1] == team and m not in dev_in], 10)
test += rng.sample([m for m in out_of_scope if m not in dev_oos], 50)

print(f"{len(in_scope)} in-scope and {len(out_of_scope)} out-of-scope test messages")
print(f"DEV:  {len(dev_in)} in scope (20 per team) + {len(dev_oos)} out of scope")
print(f"TEST: {len(test) - 50} in scope (10 per team) + 50 out of scope, none of them in DEV")

In [ ]:
# ===========================================================================
# ZERO-SHOT: load a Hugging Face zero-shot classifier for Lab 1 (heavy download, run once)
# ===========================================================================
# Why: our baseline. A zero-shot classifier sorts text into labels you name, without training
# on them: for each label it asks an NLI (natural language inference) model whether "This
# message is about <label>." follows from the message, and turns those scores into a probability per label. It runs here, in this
# runtime: no key, no API bill. The model is about 1.6 GB, so the download starts now.
# A GPU runtime (T4) is recommended for it; on CPU it still runs, only slower.
# Its numbers, like Jev's, hold for one version: we pin the model revision and print the
# library version (our numbers come from transformers 5.22.1).
import torch
import transformers
from transformers import pipeline

ZERO_SHOT_MODEL = "facebook/bart-large-mnli"
ZERO_SHOT_REVISION = "d7645e127eaf1aefc7862fd59a17a5aa8558b8ce"    # pinned, like JEV_MODEL
HAS_GPU = torch.cuda.is_available()
zero_shot = pipeline("zero-shot-classification", model=ZERO_SHOT_MODEL, revision=ZERO_SHOT_REVISION,
                     device=0 if HAS_GPU else -1)
print(f"{ZERO_SHOT_MODEL} loaded on {'GPU: ' + torch.cuda.get_device_name(0) if HAS_GPU else 'CPU'}, "
      f"transformers {transformers.__version__}.")
if not HAS_GPU:
    print("No GPU. Recommended: Runtime > Change runtime type > T4 GPU, then run Setup again. "
          "On CPU everything still works; the zero-shot cells in Labs 1 and 3 just take minutes.")

## Part 1: A confident answer is not a correct answer (about 30 min)

Every Jev answer comes with a probability, and our code acts on it. Before we let it, we check
what the numbers mean.

### 1. The problem

Follow one message through the figure below, left to right, then down: "i need to report
fraudulent activity on my usaa card", labeled banking. Module 1's team question gets about
**81%** of 200 labeled messages right (80% to 82% across our runs), but code routes one message
at a time, and the probability is what it looks at. This one came back credit_cards at
**0.99, and wrong**.

**A probability is a claim.** Until you check it on data you know, you do not know what 0.99 means.

**Figure: The problem (our run) - code trusts every answer at 0.9 or more, and nobody has measured what 0.95 means.**

<!-- DIAGRAM: p1-problem-confident-wrong -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p1-problem-confident-wrong.png" width="750" alt="STATE: the usaa fraud message, labeled banking, and 199 more labeled messages. QUESTIONS in one request: team_question_text, a few words per team, and team_question_data, 15 intents per team. JEV ANSWERS: text says credit_cards 0.99, intents says banking 0.93. YOUR CODE routes every answer at 0.9 or more. OUTCOME for the answers said 0.90 to 0.99: both questions said 0.95 on average, how often they were right is unknown, and the usaa message is a miss">

### 2. The solution

Same layout; only YOUR CODE changes: `reliability()` groups answers by stated probability and
counts the right ones. Said = right is **calibrated**; right below said is **overconfident**.
It holds for groups of answers, never one (TypeSafe's docs).

**The rule: a probability is a claim. Trust it only where you measured it, on your question, with
your model version.**

**Figure: The solution (our run) - same messages, same two questions; only YOUR CODE changes, and reliability() shows what each 0.95 means.**

<!-- DIAGRAM: p1-solution-reliability -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p1-solution-reliability.png" width="750" alt="Same layout and the same request. YOUR CODE, changed: reliability() groups answers by stated probability and counts the right ones. OUTCOME for the answers said 0.90 to 0.99: text said 0.95 and was right 0.75 of the time, overconfident; intents said 0.95 and was right every time, safe to act">

**Figure: What calibrated means - of the answers where Jev said about p, about p were right; it describes a group of answers, never one.**

<!-- DIAGRAM: what-calibrated-means -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/what-calibrated-means.png" width="590" alt="One answer at 0.95 is right or wrong and promises nothing. Collect the answers Jev said 0.90 to 0.99, mean said 0.95, and count the share that was right: below said is overconfident, as team_question_text at 0.75; equal to said is calibrated; above said is under-confident, as team_question_data at 1.00">

Read 0.95 as a claim about the group: of many answers like it, about 95 in 100 should be right.

### 3. Demo

First two helpers, because measuring takes hundreds of calls. Then the problem live: two team
questions on the same 200 DEV messages, accuracy for each, and the wrong answers that looked
sure. Then the solution: the reliability table for each question, and the plot.

In [ ]:
# ===========================================================================
# HELPER: ask Jev about many messages, and group answers by probability
# ===========================================================================
# Why: measuring means hundreds of calls. ask_all() sends them 4 at a time; reliability()
# groups answers by the probability they came with and counts how many were right.
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt

EDGES = [0.0, 0.5, 0.7, 0.9, 0.99, 1.0]   # probability groups: [0, 0.5), [0.5, 0.7), ...
# 0.99 to 1.00 is a group of its own: most answers land there, so 0.90 to 0.99 is read apart.


def ask_all(rows: list[tuple[str, str]], questions: dict, model: str = JEV_MODEL) -> list[dict]:
    """One Jev call per (message, team) row, 4 calls at a time.

    Returns one dict per row, in order: the answers, the latency in ms, the input tokens and
    the model that answered."""
    def one(row):
        start = time.perf_counter()
        response = jev.system_one(state={"message": row[0]}, model=model, questions=questions)
        return {"answers": response.answers, "ms": (time.perf_counter() - start) * 1000,
                "tokens": response.usage.input_tokens, "model": response.model}
    with ThreadPoolExecutor(max_workers=4) as pool:
        return list(pool.map(one, rows))


def top(answer) -> tuple[str, float]:
    """A Choice answer's chosen option and the probability Jev gave it."""
    return answer.choice, answer.probabilities[answer.choice]


def reliability(probabilities: list[float], correct: list[bool], name: str) -> list[tuple]:
    """Group answers by stated probability; print and return (mean said, share right, n)."""
    print(f"=== {name} ===")
    print(f"{'stated probability':>20} {'n':>4} {'said (mean)':>12} {'was right':>10}")
    groups = []
    for low, high in zip(EDGES, EDGES[1:]):
        inside = [(p, c) for p, c in zip(probabilities, correct)
                  if low <= p < high or (high == 1.0 and p == 1.0)]
        if inside:
            said = statistics.mean(p for p, _ in inside)
            right = sum(c for _, c in inside) / len(inside)
            groups.append((said, right, len(inside)))
            print(f"{f'{low:.2f} to {high:.2f}':>20} {len(inside):>4} {said:>12.2f} {right:>10.2f}")
    return groups


def plot_reliability(curves: dict) -> None:
    """curves: {name: groups from reliability()}. Points on the dashed line are calibrated."""
    plt.figure(figsize=(5.5, 5))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="calibrated: said = right")
    for name, groups in curves.items():
        said, right, n = zip(*groups)
        plt.plot(said, right, "-o", label=name)
        for x, y, k in groups:
            plt.annotate(f"n={k}", (x, y), textcoords="offset points", xytext=(5, -12), fontsize=8)
    plt.xlabel("stated probability of the answer")
    plt.ylabel("share of answers that were right")
    plt.xlim(0, 1.02)
    plt.ylim(0, 1.05)
    plt.legend(loc="upper left")
    plt.show()


print("Helpers ready: ask_all(), top(), reliability(), plot_reliability().")

In [ ]:
# ===========================================================================
# DEMO: the problem: accuracy is one number, the answers come one at a time
# ===========================================================================
# Why: two team questions over the same 200 DEV messages, both in one request per message:
# Module 1's few words per team, and Module 2's intent lists as criteria.
team_question_text = Choice(instructions="Which team should handle `message`?", criteria=TEAMS)
team_question_data = Choice(instructions="Which team should handle `message`?", criteria=TEAM_INTENTS)

part1 = ask_all(dev_in, {"text": team_question_text, "intents": team_question_data})

for name in ("text", "intents"):
    right = sum(r["answers"][name].choice == label for r, (_, label) in zip(part1, dev_in))
    print(f"team question with {name:7} descriptions: {right}/{len(dev_in)} right ({right / len(dev_in):.0%})")  # expect about 81% and 99%

print("\nText descriptions: wrong answers that came with 0.9 or more:")
for r, (text, label) in zip(part1, dev_in):
    choice, p = top(r["answers"]["text"])
    if choice != label and p >= 0.9:
        print(f"  {p:.2f}  {choice:16} (label {label:12}) | {text[:50]}")  # expect about 12 rows

**Look at:** the two accuracy lines (ours: about 81% with a few words per team, about 99% with intent lists; yours may differ by a point or two), then the misses that came with 0.9 or more: about a dozen, the usaa message among them. A high probability did not make them right.


In [ ]:
# ===========================================================================
# DEMO: the solution: group the answers by probability, count the right ones
# ===========================================================================
# Why: one row per probability group. If Jev says about 0.95 on 30 answers and is calibrated,
# about 28 or 29 of them are right. Small groups (n under 10) are noisy: read them loosely.
# Expect, in the 0.90 to 0.99 row: text right about 0.75 to 0.78, intents about 1.00.
curves = {}
for name in ("text", "intents"):
    probabilities = [top(r["answers"][name])[1] for r in part1]
    correct = [r["answers"][name].choice == label for r, (_, label) in zip(part1, dev_in)]
    curves[f"Jev, {name}"] = reliability(probabilities, correct,
                                         f"Jev, team question with {name} descriptions")
    print()

**Look at:** the `0.90 to 0.99` row of each table. Both questions said about 0.95; the few-words question was right about 0.75 to 0.78 of the time, the intent-list question about every time. Same number, different meaning: that is what calibration measures.


In [ ]:
# ===========================================================================
# DEMO: Jev's probability against real accuracy, on one chart
# ===========================================================================
# Why: the chart that answers the question. Points on the dashed line mean "said 0.8, right 80%".
# Below the line: overconfident. Above it: underconfident.
plot_reliability(curves)

# Jev also returns `confidence`. For this 10-option Choice it is the top probability rescaled
# for 10 options, (p - 1/10) / (1 - 1/10), so it ranks the answers in exactly the same order:
answer = part1[0]["answers"]["intents"]
p = top(answer)[1]
print(f"one answer: probability {p:.3f}, confidence {answer.confidence:.3f}, "
      f"(p - 1/10) / (1 - 1/10) = {(p - 0.1) / 0.9:.3f}")  # expect the last two to match

**Figure: Part 1 reliability curves (our run) - grey is what Jev said, red how often the few-words question was right, teal the intent-list question; red below grey is overconfident.**

<!-- DIAGRAM: p1-demo-curve -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p1-demo-curve.png" width="700" alt="Line chart over four groups of stated probability, 0.5 to 0.7, 0.7 to 0.9, 0.9 to 0.99 and 0.99 to 1: said 62, 81, 95 and 100 percent; few-words question right 45, 70, 75 and 97; intent-list question right 100 (3 answers only), 93, 100 and 99">

**Figure: Part 1 results (our run) - where Jev said 0.90 to 0.99, the few-words question was right 75% of the time and the intent-list question 100%; navy bars are overall accuracy.**

<!-- DIAGRAM: p1-demo-results -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p1-demo-results.png" width="700" alt="Bar chart for the answers Jev gave 0.90 to 0.99 on 200 DEV messages: few-words question said 95 and was right 75 percent of the time; intent-list question said 95 and was right 100 percent; overall accuracy 81 and 99 percent">

**Figure: The baseline Lab 1 measures (our run) - a zero-shot model puts each label into one sentence and scores it; no questions, no criteria, and its scores are a claim too.**

<!-- DIAGRAM: p1-zero-shot -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p1-zero-shot.png" width="750" alt="STATE: the usaa fraud message. Each label goes into one sentence: This example is banking, This example is work, and so on for 10 labels. The NLI model, bart-large-mnli, scores whether the message supports each sentence. SCORES sum to 1: banking 0.40, work 0.25, credit_cards 0.17. OUTCOME: top label banking, and whether that is right must be measured. In Lab 1 you choose the label text and the sentence">

NLI is natural language inference: does the message support the sentence?

In [ ]:
# ===========================================================================
# DEMO: the baseline Lab 1 measures: one message through a zero-shot model
# ===========================================================================
# Why: a zero-shot model has no questions and no criteria. It puts each label into one sentence,
# scores how well the message supports that sentence, and the scores of all labels sum to 1.
# Here the labels are the bare team names; in Lab 1 you choose better label text.
usaa = "i need to report fraudulent activity on my usaa card"
out = zero_shot(usaa, candidate_labels=list(TEAMS), hypothesis_template="This example is {}.")
for label, score in list(zip(out["labels"], out["scores"]))[:4]:
    print(f"  This example is {label + '.':20} {score:.2f}")  # expect banking on top, about 0.40
print(f"Top label: {out['labels'][0]}. The 10 scores sum to {sum(out['scores']):.2f}: they say which "
      "label won, not how often the winner is right.")

> **Think About It**: Same model, same 200 messages, same request. The answers Jev gave 0.90
> to 0.99 were right 75% to 78% of the time for the few-words question and nearly always for
> the intent-list question. Only `reliability()` could tell those two 0.95s apart. Before today,
> which would you have trusted at 0.95? And when you change a question's wording next month,
> what do you re-run?

<details><summary>Our answer</summary>

The intent-list question: its 0.90 to 0.99 answers were right every time, the few-words
question's about 3 in 4. A new wording is a new question: re-run this Part's reliability
table on the same DEV set, and re-check every threshold your code reads from it.

</details>

### 4. Lab 1: A zero-shot baseline on the same chart

Is a calibrated probability special, or does every classifier give you one? You find out with a
baseline: a Hugging Face zero-shot classifier, free, local, no training, the usual first try when
there are no labels. You measure it exactly the way we measured Jev.

**Figure: Lab 1 map - two decisions, one function, and the given code that measures it.**

<!-- DIAGRAM: p1-lab-map -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p1-lab-map.png" width="750" alt="LAB 1.1 Guided: zs_labels, the label text per team, and hypothesis, the sentence. LAB 1.2 Core: zero_shot_team(texts) makes one zero_shot call with facebook/bart-large-mnli and returns a team and probability per message. The given code runs 300 DEV messages, prints accuracy and ms per message, the reliability table, and plots it next to Jev&#x27;s curves. LAB 1.3 Challenge, optional: calibration_error, one number per curve">

#### Your Task

1. Choose the label text the zero-shot model reads for each team, and the sentence it is put in (Level 1)
2. Write `zero_shot_team()`: classify a list of messages, return the top team and its probability for each (Level 2)
3. Optional: put one number on calibration, the expected calibration error, for every curve (Level 3)

#### Level 1: Guided (about 5 min)

Fill `zs_labels` (a dictionary: team name to the label text the model reads) and `hypothesis`
(the sentence each label goes into, with `{}` where the label goes, for example
`"This message is about {}."`). The cell classifies 5 DEV messages and prints the answers.
Stuck? The FALLBACK cell below fills both so you can move on.

#### Level 2: Core (about 10 min)

Write the body of `zero_shot_team(texts)`: one call to `zero_shot(...)` for the whole list with
`candidate_labels`, `hypothesis_template` and `batch_size=16`, then map each top label back to its
team. The cell runs it on all 300 DEV messages (it takes a minute or two on a CPU), prints
accuracy and the zero-shot reliability table, and plots it next to Jev's curves.

#### Level 3: Challenge (optional) (about 10 min)

Write `calibration_error(probabilities, correct)`: the expected calibration error, the average
gap between "said" and "was right" over the groups, each group weighted by its size. Print it with
the accuracy for the three curves: Jev with intents, Jev with text, zero-shot.

#### Expected Output

- Level 1: 5 rows, a team and a score each; a few will be wrong
- Level 2: zero-shot accuracy far below Jev's (ours: 56% with the team descriptions as labels),
  a reliability curve far from the line, and the time per message on your runtime; the CHECK
  cell after it prints "Lab 1.2 passed"
- Level 3: three rows; ours: Jev intents about 0.03, Jev text about 0.10, zero-shot about 0.24

#### Hints

- Level 1: the model reads only the label text inside your sentence. Team names, `TEAMS`
  descriptions, or a team's 15 intent names joined into one string are all fair choices: try
  two, and compare their accuracy once Level 2 runs. Each team needs its own label text; the
  cell checks that they are unique
- Level 2: `zero_shot(texts, candidate_labels=..., hypothesis_template=..., batch_size=16)` returns
  one dict per text, with `"labels"` sorted best first and their `"scores"`
- Level 3: reuse the loop inside `reliability()`; the error is the sum of n times the gap, over
  all groups, divided by the number of answers

<details><summary>Stuck on Level 2? A hint</summary>

Build `label_to_team` from `zs_labels` (label text to team), as Level 1 does. Call `zero_shot()`
once on the whole list with your label texts as `candidate_labels`. For each output take
`labels[0]` and `scores[0]`, map the label back to its team, and return the (team, score) pairs in
order.

</details>

In [ ]:
# ===========================================================================
# SOLUTION: LAB 1.1 (Guided)
# ===========================================================================
# Approach:
# - The zero-shot model only sees the label text inside the hypothesis sentence, so we give it
#   the most informative text we have: Module 1's description of each team.
# Step 1: the label text the zero-shot model reads for each of the 10 teams.
zs_labels = dict(TEAMS)    # e.g. "banking" -> "accounts, balances, transfers, bills, direct deposit"

# Step 2: the sentence each label is put in; {} is replaced by the label text.
hypothesis = "This message is about {}."

assert len(set(zs_labels.values())) == len(zs_labels), "two teams share a label text: make each unique"
label_to_team = {label: team for team, label in zs_labels.items()}
for text, label in dev_in[::40]:
    out = zero_shot(text, candidate_labels=list(zs_labels.values()), hypothesis_template=hypothesis)
    print(f"{text[:44]:44} {label_to_team[out['labels'][0]]:18} {out['scores'][0]:.2f}  label {label}")
# Explanation:
# - Measured on the 200 DEV messages: team names as labels ("credit cards") got 38.5% right, the
#   descriptions 56%. More words give the NLI model more to match.
# - The pipeline's default sentence is "This example is {}."; a sentence that says what the text
#   is ("This message is about ...") reads more naturally to the model. Try both.
# - The scores are a softmax over the 10 labels: they sum to 1 and say which label won, not how
#   often the winner is right. Level 2 measures that.

In [ ]:
# ===========================================================================
# SOLUTION: LAB 1.2 (Core)
# ===========================================================================
# Approach:
# - One pipeline call for the whole list: it batches the work, which is much faster than a loop.
# - The pipeline answers with label text; map it back to the team name our labels use.
def zero_shot_team(texts: list[str]) -> list[tuple[str, float]]:
    """Classify each text into one of the 10 teams with the zero-shot pipeline.

    Returns one (team, probability) per text, in order: the top team and the score the model
    gave its label.
    """
    label_to_team = {label: team for team, label in zs_labels.items()}
    outputs = zero_shot(texts, candidate_labels=list(zs_labels.values()),
                        hypothesis_template=hypothesis, batch_size=16)
    return [(label_to_team[o["labels"][0]], o["scores"][0]) for o in outputs]


start = time.perf_counter()
zs_dev = zero_shot_team([text for text, _ in dev])
zs_ms = (time.perf_counter() - start) * 1000 / len(dev)
zs_in = zs_dev[:len(dev_in)]
correct = [team == label for (team, _), (_, label) in zip(zs_in, dev_in)]
print(f"zero-shot: {sum(correct)}/{len(dev_in)} right ({sum(correct) / len(dev_in):.0%}), "
      f"{zs_ms:.0f} ms per message on this runtime\n")
curves["zero-shot"] = reliability([p for _, p in zs_in], correct, "zero-shot, team")
plot_reliability(curves)
# Explanation:
# - Ours: 56% right, at about 120 ms per message on a laptop CPU. Its curve sits far from the
#   line: answers it scored around 0.26 were right half the time, so its numbers mean something
#   different from Jev's, and you only learn that by measuring.
# - Common mistake: comparing the zero-shot "score" with Jev's probability as if they were the
#   same kind of number. Neither is a promise until a reliability table says so.

In [ ]:
# ===========================================================================
# CHECK: Lab 1.2
# ===========================================================================
# Why: tells you whether zero_shot_team() keeps its contract before Lab 3 uses it. It never
# stops the notebook: an unfinished lab prints "not finished yet".
try:
    if zs_dev is None:
        print("Lab 1.2 not finished yet: zero_shot_team() returned None. Write its body, run Lab 1.2 again.")
    else:
        problems = []
        if len(zs_dev) != len(dev):
            problems.append(f"expected {len(dev)} results, one per DEV message, got {len(zs_dev)}")
        if not all(isinstance(row, tuple) and len(row) == 2 for row in zs_dev):
            problems.append("each result must be a (team, probability) tuple")
        else:
            not_teams = sorted({str(team) for team, _ in zs_dev if team not in TEAMS})
            if not_teams:
                problems.append(f"not team names: {not_teams[:3]} (map each label back to its team)")
            if not all(0 <= p <= 1 for _, p in zs_dev):
                problems.append("every probability must be between 0 and 1")
        if problems:
            print("Lab 1.2 not right yet: " + "; ".join(problems))
        else:
            right = sum(team == label for (team, _), (_, label) in zip(zs_dev, dev_in))
            print(f"Lab 1.2 passed: {len(zs_dev)} (team, probability) pairs, every team valid, "
                  f"{right}/{len(dev_in)} in-scope messages right.")
except Exception as check_error:      # an unfinished lab must never stop the notebook
    print(f"Lab 1.2 not finished yet: {type(check_error).__name__}: {check_error}")


In [ ]:
# ===========================================================================
# SOLUTION: LAB 1.3 (Challenge, optional)
# ===========================================================================
# Approach:
# - The same groups as reliability(); weight each group's gap by its size.
# Spec: calibration_error(probabilities, correct) returns the expected calibration error:
#   group the answers with EDGES, as reliability() does; for each group take
#   n * abs(mean said - share right); add them up and divide by the number of answers.
# Then print accuracy and calibration error for: Jev intents, Jev text, zero-shot (DEV in scope).
def calibration_error(probabilities: list[float], correct: list[bool]) -> float:
    total = 0.0
    for low, high in zip(EDGES, EDGES[1:]):
        inside = [(p, c) for p, c in zip(probabilities, correct)
                  if low <= p < high or (high == 1.0 and p == 1.0)]
        if inside:
            said = statistics.mean(p for p, _ in inside)
            right = sum(c for _, c in inside) / len(inside)
            total += len(inside) * abs(said - right)
    return total / len(probabilities)


for name, answers in [("Jev, intents", [top(r["answers"]["intents"]) for r in part1]),
                      ("Jev, text", [top(r["answers"]["text"]) for r in part1]),
                      ("zero-shot", zs_in)]:
    right = [team == label for (team, _), (_, label) in zip(answers, dev_in)]
    error = calibration_error([p for _, p in answers], right)
    print(f"{name:13} accuracy {sum(right) / len(right):4.0%}   calibration error {error:.3f}")
# Explanation:
# - Ours: Jev intents about 0.03, Jev text about 0.10, zero-shot about 0.24. Lower is better; 0
#   means every group sits on the dashed line.
# - One number hides the shape: an underconfident model and an overconfident one can score the
#   same. Keep the plot next to it. With only 5 groups and small n, treat the third decimal as noise.

## Part 2: Out of scope: when no team fits (about 25 min)

Our labeled messages so far all belong to a team. Real traffic does not: people ask an assistant
anything. This Part teaches the triage layer to say "none of ours".

### 1. The problem

CLINC150 has 1,000 **out-of-scope** messages: requests that fit none of its 150 intents, like
"what is the highest quality carpet available". **A Choice always picks one of its options**:
every one of our 100 out-of-scope DEV messages got a team, about a third of them at 0.9 or more.
A probability cut (refuse answers below about 0.8, which costs 10 in-scope messages) catches only
about half. The question "does any team fit?" was never asked.

**Figure: The problem (our run) - one request, the team question only; a Choice always picks a team, and a probability cut catches 47 of 100.**

<!-- DIAGRAM: p2-problem-choice-always-picks -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p2-problem-choice-always-picks.png" width="750" alt="STATE: the carpet message, out of scope, and i lost my credit card, labeled credit_cards. QUESTIONS in one request: team_question_data, a Choice of 10 teams; fits_question is not asked. JEV ANSWERS: carpet home 0.99, lost card credit_cards 1.00. YOUR CODE: a probability cut, below 0.80 to REVIEW, else ROUTE. OUTCOME: the carpet is routed to home, a miss, and the lost card is routed; all 100 out-of-scope messages got a team, the cut caught 47 and refused 10 in-scope messages">

### 2. The solution

Same layout, same two messages. One question is added, a **companion Noul** whose `true`
criterion is the data `TEAM_INTENTS`, as TypeSafe's guidance advises, and code decides with it. Read the cost in OUTCOME; 92 of 100 is a best
case, since the Noul sees CLINC150's exact intent list.

**The rule: a Choice cannot say "none". Ask "does anything fit?" as its own Noul, in the same
request, and decide in code.**

**Figure: The solution (our run) - the same request plus fits_question; code decides with it.**

<!-- DIAGRAM: p2-solution-companion-noul -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p2-solution-companion-noul.png" width="750" alt="Same layout and the same two messages. QUESTIONS in one request: team_question_data and the new fits_question, a companion Noul whose true criterion is TEAM_INTENTS. JEV ANSWERS: carpet home 0.99 with fits 0.05; lost card credit_cards 1.00 with fits 0.92. YOUR CODE, scope_action: fits below 0.3 OUT_OF_SCOPE, below 0.7 REVIEW, else ROUTE. OUTCOME: carpet OUT_OF_SCOPE and lost card ROUTE, both right; 92 of 100 out-of-scope messages caught; of 200 in scope, 12 flagged and 30 sent to REVIEW">

### 3. Demo

The problem live: the team question alone on the 100 out-of-scope DEV messages, and whether a
probability cut could catch them. Then the solution: the companion Noul and the team question in
one request, on all 300 DEV messages, with the speed and cost of that request. Then the rules,
in code, on three messages.

In [ ]:
# ===========================================================================
# DEMO: the problem: 100 messages no team handles, and every one gets a team
# ===========================================================================
# Why: a Choice must pick one of its options. Watch the probabilities it picks them with.
part2_oos = ask_all(dev_oos, {"team": team_question_data})
oos_top = [top(r["answers"]["team"]) for r in part2_oos]
print(f"{len(dev_oos)} out-of-scope messages, {len(oos_top)} got a team, "
      f"{sum(p >= 0.9 for _, p in oos_top)} of them at 0.9 or more. The surest ones:\n")  # expect about 30
for (text, _), (team, p) in sorted(zip(dev_oos, oos_top), key=lambda x: -x[1][1])[:6]:
    print(f"  {p:.2f}  {team:18} | {text[:56]}")

# Could a probability cut catch them? Use Part 1's in-scope answers to place the cut.
in_scope_p = sorted(top(r["answers"]["intents"])[1] for r in part1)
cut = in_scope_p[len(in_scope_p) // 20]     # 5% of in-scope answers fall below this
caught = sum(p < cut for _, p in oos_top)
print(f"\nRefuse every answer below {cut:.2f}: we lose 5% of in-scope messages and catch "
      f"{caught} of {len(dev_oos)} out-of-scope ones.")  # expect a cut near 0.80, about half caught

**Look at:** all 100 out-of-scope messages got a team, about 30 of them at 0.9 or more. The cut that costs 5% of in-scope messages sits near 0.80 and catches only about half of them.


In [ ]:
# ===========================================================================
# DEMO: the solution: a companion Noul in the same request
# ===========================================================================
# Why: the Choice answers "which team?", the Noul answers "does anything fit?". Its true
# criterion is data: the supported tasks of every team, Module 2's TEAM_INTENTS.
fits_question = Noul(
    instructions="Does `message` ask for one of the tasks our assistant supports?",
    criteria={"true": {"supported_tasks": TEAM_INTENTS},
              "false": "the message asks for something that is not one of the supported tasks"})

dev_answers = ask_all(dev, {"team": team_question_data, "fits": fits_question})
fits = [r["answers"]["fits"].noul for r in dev_answers]
groups = {"in scope": fits[:len(dev_in)], "out of scope": fits[len(dev_in):]}
print(f"{'':14} {'fits < 0.3':>10} {'0.3 to 0.7':>11} {'fits >= 0.7':>12}")
for name, values in groups.items():
    print(f"{name:14} {sum(f < 0.3 for f in values):>10} {sum(0.3 <= f < 0.7 for f in values):>11} "
          f"{sum(f >= 0.7 for f in values):>12}")

print("\nIn-scope messages the Noul calls out of scope (read them: would you?):")
for (text, label), f in zip(dev_in, groups["in scope"]):
    if f < 0.3:
        print(f"  {f:.2f}  {label:16} | {text[:52]}")

ms = statistics.median(r["ms"] for r in dev_answers)
cost = 1000 * statistics.mean(r["tokens"] for r in dev_answers) * JEV_PRICE_IN
print(f"\nThe production request, two questions: p50 {ms:.0f} ms, ${cost:.3f} per 1,000 messages.")
# expect p50 about 240 ms and about $0.09 per 1,000 messages

**Look at:** the table. Out of scope lands mostly below 0.3 (about 92 of 100); in scope mostly at 0.7 or more, with about 13 below 0.3 and about 30 in the band between. Those in-scope rows are what the Noul costs. Last line: p50 (the median call) about 240 ms, about $0.09 per 1,000 messages.


In [ ]:
# ===========================================================================
# DEMO: Jev judges, your code decides: OUT_OF_SCOPE, REVIEW or ROUTE
# ===========================================================================
# Why: the Noul thresholds from Module 1. Below 0.3 reads as no, 0.7 or more as yes, and the
# band between means the message does not settle it, so a person looks.
def scope_action(fits: float) -> str:
    if fits < 0.3:
        return "OUT_OF_SCOPE"
    if fits < 0.7:
        return "REVIEW"
    return "ROUTE"


for text in ["what are the top rated local lawyers", "i lost my credit card",
             "what is the highest quality carpet available"]:
    answers = jev.system_one(state={"message": text}, model=JEV_MODEL,
                             questions={"team": team_question_data, "fits": fits_question}).answers
    team, p = top(answers["team"])
    print(f"{text[:44]:44} team {team:18} {p:.2f}  fits {answers['fits'].noul:.2f}  "
          f"-> {scope_action(answers['fits'].noul)}")  # expect OUT_OF_SCOPE, ROUTE, OUT_OF_SCOPE

**Figure: Part 2 results (our run) - out of 100 out-of-scope messages, the cut catches 47 and the companion Noul 92.**

<!-- DIAGRAM: p2-demo-results -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p2-demo-results.png" width="700" alt="Bar chart of out-of-scope DEV messages caught out of 100: the probability cut 47, the companion Noul 92">

**Figure: Part 2 cost (our run) - what the Noul costs in scope: 42 messages wait for a person, against 10 for the cut.**

<!-- DIAGRAM: p2-demo-cost -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p2-demo-cost.png" width="700" alt="Stacked bar chart of in-scope DEV messages not routed, out of 200: the probability cut refuses 10; the Noul flags 12 as out of scope and sends 30 more to REVIEW, 42 in all">

The Noul is not free: in our run 42 in-scope messages (12 flagged out of scope + 30 in the band)
wait for a person, against 10 for the cut. It buys 45 more out-of-scope catches (92 against 47)
for 32 more in-scope messages sent to a person. The request costs one call: p50 (the median call) about
240 ms, $0.09 per 1,000 messages.

> **Think About It**: The team answer for the carpet message did not change: still home, at
> 0.98 or more. We did not make the Choice smarter; we asked a second question it could never answer.
> Now read the in-scope messages the Noul flagged: some are messages a person would also hesitate
> on, and a few CLINC labels are arguable. When the Noul and the label disagree, who is wrong:
> the question, the label, or neither?

<details><summary>Our answer</summary>

Often neither: the message is ambiguous, and a REVIEW is the right outcome. When the label is
clearly right, the Noul's criteria miss a task: fix the criteria, not the threshold. When the
label is arguable, count it apart; it is not a Jev error.

</details>

### 4. Lab 2: Out of scope for your product

"Out of scope" is not a property of a message: it depends on what **your** product supports.
Say our assistant launches with only three teams: banking, credit_cards and travel. A message
about a dead car battery is now out of scope, and a 3-option Choice will still pick one of the
three. You build the triage for the launch.

**Figure: Lab 2 map - what is given, the two criteria you write, and the rules launch_triage() applies.**

<!-- DIAGRAM: p2-lab-map -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p2-lab-map.png" width="750" alt="GIVEN: LAUNCH_TEAMS banking, credit_cards and travel with their 45 intents, and launch_team_question, a Choice of the 3. LAB 2.1 Guided: scope_true and scope_false make launch_fits_question, a Noul. LAB 2.2 Core: launch_triage makes one jev.system_one call with team and fits, then fits below 0.3 is OUT_OF_SCOPE, fits below 0.7 is REVIEW, team probability below 0.7 is REVIEW, otherwise ROUTE. LAB 2.3 Challenge, optional: none_option_check compares a none_of_these option with the Noul">

#### Your Task

1. Write the `true` and `false` criteria of the launch scope's companion Noul (Level 1)
2. Write `launch_triage()`: one request with both questions, then the rules in code (Level 2)
3. Optional: compare the companion Noul with a "none_of_these" option inside the Choice (Level 3)

#### Level 1: Guided (about 5 min)

`launch_scope` (the three launch teams and their intents) and `launch_team_question` (a Choice
of the three) are given. Fill `scope_true` (what a supported message asks for; Module 2's lesson
applies) and `scope_false` (a concrete situation for "no"). The cell asks both questions about
six messages. Stuck? The FALLBACK cell below fills both.

#### Level 2: Core (about 10 min)

Write the body of `launch_triage(message)`: one `jev.system_one` call with `launch_team_question`
and `launch_fits_question`, then these rules, checked in this order:

| Rule | Action |
|---|---|
| fits < 0.3 | OUT_OF_SCOPE |
| fits < 0.7 | REVIEW: the message does not settle it |
| team probability < 0.7 | REVIEW: in scope, but torn between teams |
| otherwise | ROUTE to the team |

The cell runs 30 CLINC150 messages, 15 from the launch teams and 15 from the other seven, and
prints a table of actions per group.

#### Level 3: Challenge (optional) (about 10 min)

Write `none_option_check(rows)`: ask, in one request per message, your `launch_fits_question` and
a Choice of the three teams plus a fourth option, `none_of_these`. Count, for each approach, the
launch messages kept and the other messages caught. TypeSafe's guidance says that, as of Jev
1.14, the companion Noul usually works better than a "none of the above" option. Does it here?

#### Expected Output

- Level 1: six rows; the three launch messages fit (about 0.9 or more), the three others do not
  (below 0.1), even though the Choice gives each of them a launch team, often at 1.00
- Level 2: in our runs, all 15 launch messages ROUTE; of the 15 others, 13 or 14 OUT_OF_SCOPE
  and the rest REVIEW, none routed; the CHECK cell after it prints "Lab 2.2 passed"
- Level 3: two rows; ours: Noul 15 kept and 13 or 14 caught, none option 15 kept and 12 caught

#### Hints

- Level 1: a criterion can be a sentence or data. Module 2 showed which one draws the line where
  the product does; `launch_scope` is already built. For `false`, describe what people ask that
  you do not support as a concrete situation, not "not supported"
- Level 2: `top(response.answers["team"])` gives the team and its probability;
  `response.answers["fits"].noul` is the Noul
- Level 3: `{**launch_scope, "none_of_these": "..."}` adds the fourth option

<details><summary>Stuck on Level 2? A hint</summary>

One `jev.system_one` call with `questions={"team": launch_team_question, "fits": launch_fits_question}`.
Read the team and its probability with `top()` and the Noul with `.noul`. Then one `if / elif`
chain in the table's order: the first rule that matches sets the action. Return a dict with the
four keys.

</details>

In [ ]:
# ===========================================================================
# SOLUTION: LAB 2.1 (Guided)
# ===========================================================================
# Approach:
# - "true" is data: the 45 intents the launch supports, the way Module 2 described teams.
# - "false" is a concrete situation, as TypeSafe's guidance advises, not "not supported".
LAUNCH_TEAMS = ["banking", "credit_cards", "travel"]
launch_scope = {team: TEAM_INTENTS[team] for team in LAUNCH_TEAMS}     # 45 intents
launch_team_question = Choice(instructions="Which team should handle `message`?", criteria=launch_scope)

# Step 1: what a message we support asks for (the Noul's "true").
scope_true = {"supported_tasks": launch_scope}
# Step 2: a concrete situation for "no" (the Noul's "false").
scope_false = ("the message is about cooking, the weather, music, cars, work, small talk "
               "or the assistant itself")

launch_fits_question = Noul(
    instructions="Does `message` ask for one of the tasks our assistant supports?",
    criteria={"true": scope_true, "false": scope_false})
for text in ["i lost my credit card", "book me a hotel in paris", "what's my checking balance",
             "my car battery is dead", "add eggs to my shopping list", "what's the weather tomorrow"]:
    answers = jev.system_one(state={"message": text}, model=JEV_MODEL, questions={
        "team": launch_team_question, "fits": launch_fits_question}).answers
    team, p = top(answers["team"])
    print(f"{text:32} team {team:13} {p:.2f}   fits {answers['fits'].noul:.2f}")
# Explanation:
# - The dead car battery goes to travel at 1.00 and fits at 0.02: the Choice had three options
#   and had to pick one. The Noul is the only thing that says "none".
# - A few words instead of data ("Is `message` about banking, credit cards or travel?") let
#   "book an uber" and "traffic on my route" through at 0.9 or more in our probe: travel-sounding
#   words, wrong scope. The intent lists draw the line where the product does.
# - Be honest about the test: our "false" names the very topics Lab 2.2's other messages come
#   from, so its catch rate flatters it. Check any criterion on messages you did not write it
#   from (CLINC150's out-of-scope set is a good start).

In [ ]:
# ===========================================================================
# SOLUTION: LAB 2.2 (Core)
# ===========================================================================
# Approach:
# - One request, two judgments; the rules are plain Python, checked in order, so the scope rule
#   wins over the team rule.
LAB2_MESSAGES = (random.Random(5).sample([m for m in in_scope if m[1] in LAUNCH_TEAMS], 15)
                 + random.Random(5).sample([m for m in in_scope if m[1] not in LAUNCH_TEAMS], 15))


def launch_triage(message: str) -> dict:
    """One Jev call with launch_team_question and launch_fits_question.

    Returns {"team", "team_probability", "fits", "action"}, the action from the rules table in
    the lab text.
    """
    response = jev.system_one(state={"message": message}, model=JEV_MODEL, questions={
        "team": launch_team_question, "fits": launch_fits_question})
    team, team_probability = top(response.answers["team"])
    fits = response.answers["fits"].noul
    if fits < 0.3:
        action = "OUT_OF_SCOPE"
    elif fits < 0.7:
        action = "REVIEW"
    elif team_probability < 0.7:
        action = "REVIEW"
    else:
        action = "ROUTE"
    return {"team": team, "team_probability": team_probability, "fits": fits, "action": action}


counts = {}
for text, label in LAB2_MESSAGES:
    result = launch_triage(text)
    group = "launch team" if label in LAUNCH_TEAMS else "other team"
    counts[(group, result["action"])] = counts.get((group, result["action"]), 0) + 1
    if (group == "launch team") != (result["action"] == "ROUTE"):
        print(f"  read this one: {result['action']:12} fits {result['fits']:.2f} "
              f"{result['team']:13} (label {label}) | {text[:40]}")

print(f"\n{'':12} {'ROUTE':>6} {'REVIEW':>7} {'OUT_OF_SCOPE':>13}")
for group in ("launch team", "other team"):
    print(f"{group:12} {counts.get((group, 'ROUTE'), 0):>6} {counts.get((group, 'REVIEW'), 0):>7} "
          f"{counts.get((group, 'OUT_OF_SCOPE'), 0):>13}")
# Explanation:
# - Ours: 15 of 15 launch messages ROUTE; of the other 15, 13 or 14 OUT_OF_SCOPE and the rest
#   REVIEW, none routed. The REVIEW rows sit in the band where the message does not settle it.
# - Every one of the 15 other messages still got a launch team from the Choice. Without the
#   Noul, all 15 would have been routed to banking, credit_cards or travel.
# - Common mistake: checking the team probability first. A sure team (travel 1.00) says nothing
#   about scope.

In [ ]:
# ===========================================================================
# CHECK: Lab 2.2
# ===========================================================================
# Why: tells you whether launch_triage() keeps its contract. One Jev call. It never stops the
# notebook: an unfinished lab prints "not finished yet".
try:
    result = launch_triage("i lost my credit card")
    if result is None:
        print("Lab 2.2 not finished yet: launch_triage() returned None. Write its body, run Lab 2.2 again.")
    else:
        problems = []
        missing = {"team", "team_probability", "fits", "action"} - set(result)
        if missing:
            problems.append(f"missing keys {sorted(missing)}")
        elif result["action"] not in ("ROUTE", "REVIEW", "OUT_OF_SCOPE"):
            problems.append(f"action {result['action']!r} is not ROUTE, REVIEW or OUT_OF_SCOPE")
        elif result["team"] not in LAUNCH_TEAMS:
            problems.append(f"team {result['team']!r} is not a launch team")
        if sum(counts.values()) != len(LAB2_MESSAGES):
            problems.append(f"the Lab 2.2 table counts {sum(counts.values())} of {len(LAB2_MESSAGES)} "
                            "messages: run Lab 2.2 again")
        launch_out = counts.get(("launch team", "OUT_OF_SCOPE"), 0)
        other_routed = counts.get(("other team", "ROUTE"), 0)
        if launch_out > 3:
            problems.append(f"{launch_out} of 15 launch messages are OUT_OF_SCOPE: check scope_true")
        if other_routed > 3:
            problems.append(f"{other_routed} of 15 other messages ROUTE: does the fits rule come first?")
        if problems:
            print("Lab 2.2 not right yet: " + "; ".join(problems))
        else:
            print(f"Lab 2.2 passed: the four keys, a valid action ({result['action']} for the lost card), "
                  f"{15 - launch_out} of 15 launch messages kept, {other_routed} other messages routed.")
except Exception as check_error:      # an unfinished lab must never stop the notebook
    print(f"Lab 2.2 not finished yet: {type(check_error).__name__}: {check_error}")


In [ ]:
# ===========================================================================
# SOLUTION: LAB 2.3 (Challenge, optional)
# ===========================================================================
# Approach:
# - Both approaches in ONE request per message, so they see the exact same state.
# Spec: none_option_check(rows) asks, in ONE request per (message, team) row,
#   "fits": launch_fits_question, and
#   "team": a Choice of launch_scope plus a fourth option "none_of_these".
# For each approach it prints: launch messages kept (fits >= 0.3 / not none_of_these) out of 15,
# and other messages caught (fits < 0.3 / none_of_these) out of 15.
def none_option_check(rows: list[tuple[str, str]]) -> None:
    with_none = Choice(instructions="Which team should handle `message`?",
                       criteria={**launch_scope,
                                 "none_of_these": "the message asks for something none of these teams handles"})
    kept = {"noul": 0, "none option": 0}
    caught = {"noul": 0, "none option": 0}
    for text, label in rows:
        answers = jev.system_one(state={"message": text}, model=JEV_MODEL,
                                 questions={"fits": launch_fits_question, "team": with_none}).answers
        says_out = {"noul": answers["fits"].noul < 0.3,
                    "none option": answers["team"].choice == "none_of_these"}
        for way, out in says_out.items():
            if label in LAUNCH_TEAMS:
                kept[way] += not out
            else:
                caught[way] += out
    for way in kept:
        print(f"{way:12} launch messages kept {kept[way]}/15   other messages caught {caught[way]}/15")


none_option_check(LAB2_MESSAGES)
# Explanation:
# - Ours: Noul 15 kept, 13 or 14 caught; none option 15 kept, 12 caught. Close on 30 messages. On
#   Part 2's 300 DEV messages (10 teams), the demo's Noul caught 92 of 100 and flagged 12 to 15
#   of 200 in scope; a none option, in our planning probe, caught 80 and lost 5. A trade, not a
#   clean win.
# - The Noul also gives you a band (0.3 to 0.7) for REVIEW; the none option gives one more
#   probability inside a distribution, mixed with the teams. That is why the guidance prefers the
#   Noul, and why you measure on your own data before you choose.

## Part 3: Choosing the review threshold from data (about 30 min)

We now have two numbers per message: how sure Jev is of the team, and how likely the message
fits at all. Somewhere a line splits "route it" from "a person looks". This Part puts that line
where the data says.

### 1. The problem

One number to gate on, composed in code as in Module 2: `route_score = team probability x fits`.
It is a **score**, not a checked probability (nobody measured `fits` for calibration), and that
is fine: this Part never trusts its value, only what it does on labeled data. The figure follows
two guessed lines over the same 300 DEV messages: Module 2's placeholder `REVIEW_BELOW` of 0.5,
and a 0.9 that looks safer. Each trades customer-facing errors against a
person's time, and **nobody said which trade we want**.

**Figure: The problem (our run) - two guessed thresholds, two different prices, and no target to choose between them.**

<!-- DIAGRAM: p3-problem-guessed-threshold -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p3-problem-guessed-threshold.png" width="750" alt="STATE: 300 DEV messages, 200 in scope and 100 out of scope. QUESTIONS: team_question_data and fits_question. JEV ANSWERS composed in code: route_score is team probability times fits, 0.92 for the lost card and 0.05 for the carpet. YOUR CODE guesses: 0.5, Module 2&#x27;s placeholder, and 0.9, which looks safer. OUTCOME on DEV: 0.5 routes 181 with 2.8 percent wrong and 119 to a person; 0.9 routes 112 with 0.9 percent wrong and 188 to a person; no target was said, so the price is unknown">

### 2. The solution

Same layout; only YOUR CODE changes: a target in product words, a sweep on DEV, the
**lowest** passing threshold (every step up sends more messages to a person), one check on
TEST. With 0 wrong in n routed, the true error rate is probably below 3/n: the **rule of
three**. With k wrong, the **Wilson interval** gives the highest error rate still plausible
from k wrong in n, at 95%. It holds for this traffic mix (a third out of
scope) and the pinned model only; never nudge it on TEST.

**The rule: a threshold is a price. State the target, choose on DEV, check once on TEST, and pin
the model.**

**Figure: The solution (our run) - a target said out loud, a sweep on DEV, one check on TEST, a pinned model.**

<!-- DIAGRAM: p3-solution-threshold-from-data -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p3-solution-threshold-from-data.png" width="750" alt="Same layout. YOUR CODE, changed: a 2 percent target and a sweep on DEV; 0.5 has 2.8 percent wrong and misses, 0.7 has 1.3 percent and is the lowest that passes, 0.9 has 0.9 percent and sends 44 more to a person. OUTCOME on DEV at 0.7: 156 routed, 1.3 percent wrong, 144 to a person. Then TEST once at 0.7: 78 of 150 routed with 0 wrong, a true rate under 3.8 percent. Pinned: jev-1.14.0">

### 3. Demo

We reuse the 300 DEV answers from Part 2: no new calls until the last cell. The problem live:
the two guesses and their price. Then the solution: the sweep, the pick, and one check on 150
TEST messages we have not touched yet.

In [ ]:
# ===========================================================================
# DEMO: the problem: two guessed thresholds, two different prices
# ===========================================================================
# Why: same messages, same answers, only the line moves. Count what each line costs.
def route_score(answers) -> float:
    """A score for "in scope AND the right team": team x fits, composed in code.

    Not a checked probability: we choose its threshold from labeled data, never from its value."""
    return top(answers["team"])[1] * answers["fits"].noul


def price(threshold: float, scores: list[float], correct: list[bool]) -> tuple[int, int]:
    """Route every message whose score is at least threshold. Returns (routed, wrong among routed)."""
    routed = [c for s, c in zip(scores, correct) if s >= threshold]
    return len(routed), routed.count(False)


dev_scores = [route_score(r["answers"]) for r in dev_answers]
dev_correct = [label != "oos" and r["answers"]["team"].choice == label    # routing it is right
               for r, (_, label) in zip(dev_answers, dev)]

for guess in (0.5, 0.9):
    routed, wrong = price(guess, dev_scores, dev_correct)
    print(f"guess {guess}: route {routed} of {len(dev)}, {wrong} of them wrong ({wrong / routed:.1%}), "
          f"{len(dev) - routed} to a person")  # expect about 180 and about 110 routed

**Look at:** ours, 0.5 routes about 180 of 300 with about 2.8% wrong; 0.9 routes about 110 with about 0.9% wrong and sends the rest to a person. A routed out-of-scope message counts as wrong: no team should get it.


In [ ]:
# ===========================================================================
# DEMO: the solution: sweep the threshold on DEV, pick it from the target
# ===========================================================================
# Why: the target first, in words. Then every candidate line, and the lowest one that meets it.
import math


def wilson_upper(wrong: int, n: int, z: float = 1.96) -> float:
    """Upper end of the 95% Wilson interval for an error rate of wrong out of n."""
    p = wrong / n
    return (p + z * z / (2 * n) + z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n))) / (1 + z * z / n)


TARGET_ERROR = 0.02     # at most 2 wrong in every 100 routed messages
print(f"{'threshold':>9} {'routed':>7} {'wrong':>6} {'error':>6} {'to a person':>12}")
sweep = []
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95]:
    routed, wrong = price(threshold, dev_scores, dev_correct)
    sweep.append((threshold, routed, wrong))
    print(f"{threshold:>9.2f} {routed:>7} {wrong:>6} {wrong / routed:>6.1%} {len(dev) - routed:>12}")
chosen = next((t for t, routed, wrong in sweep if routed and wrong / routed <= TARGET_ERROR), None)
if chosen is None:
    print(f"\nNo threshold meets {TARGET_ERROR:.0%} on DEV: everything goes to a person.")
else:
    routed, wrong = price(chosen, dev_scores, dev_correct)
    print(f"\nLowest threshold with at most {TARGET_ERROR:.0%} wrong on DEV: {chosen}")  # expect 0.7
    print(f"  {wrong} wrong of {routed} routed ({wrong / routed:.1%}). With only {routed} messages the true "
          f"rate could be as high as {wilson_upper(wrong, routed):.1%} (95% Wilson interval).")

**Figure: Part 3 sweep (our run) - each step up routes fewer messages; the pick, 0.7, keeps 156 of 300.**

<!-- DIAGRAM: p3-demo-sweep -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p3-demo-sweep.png" width="700" alt="Bar chart of DEV messages routed out of 300 per threshold: 0.3 194, 0.4 188, 0.5 181, 0.6 171, 0.7 156 and picked, 0.8 142, 0.9 112, 0.95 85">

**Figure: Part 3 error (our run) - share wrong per threshold against the 2% line; 0.7 is the lowest step under it. The bumps (0.95 above 0.8) are 1 or 2 messages: small n, noise.**

<!-- DIAGRAM: p3-demo-error -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p3-demo-error.png" width="700" alt="Bar chart of the percent wrong among routed DEV messages per threshold, with a line at the 2 percent target: 0.3 4.6, 0.4 2.7, 0.5 2.8, 0.6 2.9, 0.7 1.3 and picked, 0.8 0.7, 0.9 0.9, 0.95 1.2">

In [ ]:
# ===========================================================================
# DEMO: check the threshold once, on TEST: messages it was not chosen on
# ===========================================================================
# Why: DEV picked the line, so DEV's numbers flatter it. TEST is the honest check, done once
# here; Lab 3 reuses TEST only to report, never to choose.
threshold = chosen if chosen is not None else 1.01      # no threshold: route nothing
start = time.perf_counter()
test_answers = ask_all(test, {"team": team_question_data, "fits": fits_question})
jev_ms_per_message = (time.perf_counter() - start) * 1000 / len(test)   # throughput, 4 at a time
test_scores = [route_score(r["answers"]) for r in test_answers]
test_correct = [label != "oos" and r["answers"]["team"].choice == label
                for r, (_, label) in zip(test_answers, test)]
routed, wrong = price(threshold, test_scores, test_correct)
print(f"TEST, threshold {threshold} chosen on DEV, answered by {test_answers[0]['model']}:")
print(f"  routed {routed} of {len(test)}, {wrong} wrong, {len(test) - routed} to a person")  # expect about 78, 0 wrong
if routed and wrong == 0:
    # Rule of three: 0 errors in n tries means the true rate is below 3/n, with 95% confidence.
    bound = 3 / routed
    print(f"  0 wrong of {routed}: the true error rate is probably under {bound:.1%} (rule of three)")
    if bound > TARGET_ERROR:
        print(f"  Verdict: consistent with {TARGET_ERROR:.0%}, but TEST is too small to prove it; "
              f"proving it takes about {3 / TARGET_ERROR:.0f} routed messages with 0 wrong.")


def final_action(answers, threshold: float) -> str:
    """The whole triage layer: Part 2's scope rule and Part 3's line, in one place."""
    # Lab 2 checks fits first. Here route_score already multiplies by fits, so a low fits never routes.
    if route_score(answers) >= threshold:
        return "ROUTE"
    if answers["fits"].noul < 0.3:    # 0.3 is still Module 1's guess: tune it the same way
        return "OUT_OF_SCOPE"   # a person confirms, then sends the polite "we can't help"
    return "REVIEW"


actions = [final_action(r["answers"], threshold) for r in test_answers]
print("  final actions on TEST:", {a: actions.count(a) for a in ("ROUTE", "REVIEW", "OUT_OF_SCOPE")})
print(f"  Jev throughput: {jev_ms_per_message:.0f} ms per message (4 calls at a time)")

**Figure: Part 3 check (our run) - at 0.7, what DEV and TEST measured and how high the true error could be, against the 2% target.**

<!-- DIAGRAM: p3-demo-test -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p3-demo-test.png" width="700" alt="Bar chart at threshold 0.7, percent wrong among routed with a line at the 2 percent target: DEV 2 of 156, 1.3; DEV 95 percent Wilson upper end 4.6; TEST 0 of 78, 0; TEST rule-of-three bound 3.8. Both upper ends sit above 2 percent: consistent with the target, not proof of it">

**Figure: The whole triage layer (our run) - final_action() puts Part 2's scope rule and Part 3's line in one place.**

<!-- DIAGRAM: p3-demo-final-action -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p3-demo-final-action.png" width="420" alt="Rule tree: route_score is team probability times fits. If route_score is at least 0.7, ROUTE to the team, 78 TEST messages. Otherwise, if fits is below 0.3, OUT_OF_SCOPE and a person confirms, 56. Otherwise REVIEW and a person routes it, 16">

> **Think About It**: The threshold came from 300 messages: in our run 2 wrong of 156 routed on
> DEV, which could still mean a true rate as high as 4.6%. On TEST it held, 0 wrong of 78 routed,
> and the rule of three says "probably under 3.8%": not "never", and not yet our 2%. How many
> labeled messages would you want before you ship it? And which of today's numbers would you
> re-measure the day you move from `jev-1.14.0` to the next release?

<details><summary>Our answer</summary>

To back 2% with 0 wrong, the rule of three needs about 3 / 0.02 = 150 routed messages; at
about half routed, that is about 300 labeled TEST messages. On a new version, re-measure all
of it: the reliability tables, the DEV sweep and its pick, the TEST check, and the Noul's
0.3 / 0.7 bands.

</details>

In [ ]:
# ===========================================================================
# BONUS DEMO: a trained baseline, when you have labels (optional)
# ===========================================================================
# Why: Module 1's lesson, measured here too. If you have thousands of labeled messages for a
# fixed set of teams, train a classifier: CLINC150 ships 15,000. Jev is for judgments you have
# no labels for. Optional: skip it and nothing breaks; if you run it, Lab 3.2 adds its row.
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

clinc_train = load_dataset("clinc/clinc_oos", "plus", split="train")
train = [(row["text"], intent_to_team[intent_names[row["intent"]]])
         for row in clinc_train if intent_names[row["intent"]] != "oos"]
tfidf = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True),
                      LogisticRegression(max_iter=2000))
tfidf.fit([text for text, _ in train], [team for _, team in train])


def tfidf_team(texts: list[str]) -> list[tuple[str, float]]:
    """Top team and its probability per text, from the trained classifier."""
    return [(tfidf.classes_[p.argmax()], float(p.max())) for p in tfidf.predict_proba(texts)]


tfidf_dev = tfidf_team([text for text, _ in dev])
correct = [team == label for (team, _), (_, label) in zip(tfidf_dev[:len(dev_in)], dev_in)]
print(f"Trained on {len(train):,} labeled messages. DEV in scope: {sum(correct)}/{len(dev_in)} right\n")  # expect about 94%
curves["TF-IDF, trained"] = reliability([p for _, p in tfidf_dev[:len(dev_in)]], correct,
                                        "TF-IDF + LogisticRegression, team")
print(f"\nOut-of-scope DEV messages it gives 0.9 or more: "
      f"{sum(p >= 0.9 for _, p in tfidf_dev[len(dev_in):])} of {len(dev_oos)}")

### 4. Lab 3: A review threshold for Jev and for the baseline

This is the module's main lab. You turn the demo's sweep into a function
that chooses a threshold from your target, then run it on Jev and on Lab 1's zero-shot baseline,
and compare them on what matters: how many messages each can route on its own, how many of those
are wrong, how fast and at what cost.

**Figure: Lab 3 map - your two decisions feed one function, and the given code runs it on both models.**

<!-- DIAGRAM: p3-lab-map -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/p3-lab-map.png" width="750" alt="LAB 3.1 Guided: MAX_ERROR, the share of routed messages that may be wrong, and MIN_ROUTED, the least DEV evidence you trust. LAB 3.2 Core: choose_threshold tries every distinct score lowest first and returns the first that routes at least min_routed with error at most max_error, else None. The given code runs it for Jev on route_score and for the zero-shot model from Lab 1 on its top probability, DEV to threshold to TEST, and prints one table: threshold, routed, wrong, to a person, ms, dollars. LAB 3.3 Challenge, optional: version_check, the same threshold on jev-1.13.0">

#### Your Task

1. Choose the target and the smallest amount of evidence you accept (Level 1)
2. Write `choose_threshold()`: the lowest threshold that meets the target on DEV (Level 2)
3. Optional: check whether the threshold survives a model version change (Level 3)

#### Level 1: Guided (about 5 min)

Fill `MAX_ERROR` (the largest share of routed messages that may be wrong) and `MIN_ROUTED` (a
threshold that routes fewer DEV messages than this is not trusted, because a handful of messages
proves nothing). Write one comment per value saying why. The cell lists the thresholds that meet
your target on DEV for Jev and for the zero-shot model. Stuck? The FALLBACK cell fills both.

#### Level 2: Core (about 10 min)

Write the body of `choose_threshold(scores, correct, max_error, min_routed)`: try every distinct
score as a threshold, lowest first; return the first one whose routed messages are at least
`min_routed` and whose error among them is at most `max_error`; return `None` when no threshold
qualifies. The cell checks it on a tiny example, then picks a threshold for Jev (score:
`route_score`) and for the zero-shot model (score: its top probability) on DEV, applies
both to TEST, and prints one table (threshold, routed, wrong, sent to a person, ms per message,
$ per 1,000) and the same numbers as a chart. If you ran the optional BONUS DEMO, a third row
shows the trained TF-IDF classifier.

#### Level 3: Challenge (optional) (about 10 min)

Write `version_check(old_model)`: ask the TEST messages again with `jev-1.13.0`, apply the
threshold you chose for `jev-1.14.0`, and print routed and wrong for both versions, plus how many
team answers changed. Does the threshold transfer? Old versions can be retired: if `jev-1.13.0`
no longer answers, catch `TypeSafeError` (from `typesafe_sdk`) and say so.

#### Expected Output

- Level 1: for Jev, several thresholds from about 0.65 or 0.7 up meet a 2% target; for the
  zero-shot model, none
- Level 2: the check passes, then a table and a chart; ours: Jev routes 78 to 80 of 150, 0 wrong
- Level 2: zero-shot gets no threshold, so a person reads all 150
- Level 3: ours, 1.13 routed 73 to 77 with 0 wrong; 4 to 8 of 150 team answers changed

#### Hints

- Level 1: what does a wrong route cost you, and what does a person's minute cost? A bank
  might pick 1%, a recipe app 5%
- Level 2: `sorted(set(scores))` gives the candidates; `price(threshold, scores, correct)` from the
  demo returns (routed, wrong). Routed only shrinks as the threshold grows, so once it is below
  `min_routed` you can stop
- Level 2: the ms column is one measure for all rows, wall time per message over the TEST batch
  (Jev: 4 calls at a time over the network; zero-shot: batches of 16 on your runtime). Ours on a
  laptop: Jev about 62 ms, zero-shot about 120 ms; TF-IDF (BONUS) under 1 ms, 37 routed
- Level 3: `ask_all(test, questions, model="jev-1.13.0")`

<details><summary>Stuck on Level 2? A hint</summary>

Loop over `sorted(set(scores))`. For each threshold call `price()`. If routed is below
`min_routed`, return `None` (every higher threshold routes fewer). If `wrong / routed` is at most
`max_error`, return that threshold. After the loop, return `None`.

</details>

In [ ]:
# ===========================================================================
# SOLUTION: LAB 3.1 (Guided)
# ===========================================================================
# Approach:
# - The target is a product decision: a wrong route costs a customer a bounce between teams;
#   a review costs a person a minute. For a support desk, 2 in 100 is a fair trade.
# - A threshold backed by a handful of messages is luck, so we demand at least 30.
# Step 1: your target. At most this share of routed messages may be wrong. Say why.
MAX_ERROR = 0.02   # 2 bounced customers in 100 routed is what our support desk accepts
# Step 2: the least evidence you accept: a threshold must route at least this many DEV messages.
MIN_ROUTED = 30    # below 30, one wrong message moves the error by more than 3 points

candidates = [round(0.05 * i, 2) for i in range(2, 20)]       # 0.10, 0.15, ..., 0.95
models = {"Jev": (dev_scores, dev_correct)}
if zs_dev is not None:
    models["zero-shot"] = ([p for _, p in zs_dev],
                           [label != "oos" and team == label for (team, _), (_, label) in zip(zs_dev, dev)])
for name, (scores, correct) in models.items():
    passing = []
    for t in candidates:
        routed, wrong = price(t, scores, correct)
        if routed and routed >= MIN_ROUTED and wrong / routed <= MAX_ERROR:
            passing.append(t)
    print(f"{name:10} {len(passing)} of {len(candidates)} thresholds meet your target on DEV: {passing}")
# Explanation:
# - Ours: Jev, 6 or 7 thresholds from 0.65 or 0.70 up to 0.95; zero-shot, none: its top scores
#   rarely pass 0.7, and the routed set always mixes in wrong teams and out-of-scope messages.
# - Try MAX_ERROR = 0.01: the Jev list shrinks to the top thresholds and more messages go to a
#   person. That is the price of the stricter target, now visible.

In [ ]:
# ===========================================================================
# SOLUTION: LAB 3.2 (Core)
# ===========================================================================
# Approach:
# - Candidates are the scores themselves, lowest first: the first one that meets the target
#   routes the most messages. Routed only shrinks as the threshold grows, so we can stop early.
def choose_threshold(scores: list[float], correct: list[bool], max_error: float,
                     min_routed: int) -> float | None:
    """The lowest threshold that meets the target on these labeled scores.

    Tries every distinct score as a threshold, lowest first. Returns the first threshold whose
    routed messages (score >= threshold) number at least min_routed and are wrong at most
    max_error of the time. Returns None when no threshold qualifies.
    """
    for threshold in sorted(set(scores)):
        routed, wrong = price(threshold, scores, correct)
        if routed < min_routed:
            return None          # every higher threshold routes even fewer
        if wrong / routed <= max_error:
            return threshold
    return None


jev_threshold = None
checks = [choose_threshold([0.95, 0.9, 0.8, 0.6], [True, True, True, False], 0.0, 2),   # 0.8
          choose_threshold([0.95, 0.9, 0.8, 0.6], [True, True, True, False], 0.0, 4),   # None
          choose_threshold([0.95, 0.9, 0.8, 0.6], [False, True, True, True], 0.0, 1)]   # None
assert checks == [0.8, None, None], f"expected [0.8, None, None], got {checks}"
rows = []
jev_threshold = choose_threshold(dev_scores, dev_correct, MAX_ERROR, MIN_ROUTED)
rows.append(("Jev", jev_threshold, test_scores, test_correct, jev_ms_per_message,
             1000 * statistics.mean(r["tokens"] for r in test_answers) * JEV_PRICE_IN))
zs_threshold = choose_threshold([p for _, p in zs_dev],
                                [label != "oos" and team == label
                                 for (team, _), (_, label) in zip(zs_dev, dev)],
                                MAX_ERROR, MIN_ROUTED)
start = time.perf_counter()
zs_test = zero_shot_team([text for text, _ in test])
zs_ms_test = (time.perf_counter() - start) * 1000 / len(test)
rows.append(("zero-shot", zs_threshold, [p for _, p in zs_test],
             [label != "oos" and team == label for (team, _), (_, label) in zip(zs_test, test)],
             zs_ms_test, 0.0))
if globals().get("tfidf_dev") is not None:      # the optional BONUS DEMO ran
    tf_threshold = choose_threshold([p for _, p in tfidf_dev],
                                    [label != "oos" and team == label
                                     for (team, _), (_, label) in zip(tfidf_dev, dev)],
                                    MAX_ERROR, MIN_ROUTED)
    start = time.perf_counter()
    tfidf_test = tfidf_team([text for text, _ in test])
    tf_ms = (time.perf_counter() - start) * 1000 / len(test)
    rows.append(("TF-IDF", tf_threshold, [p for _, p in tfidf_test],
                 [label != "oos" and team == label for (team, _), (_, label) in zip(tfidf_test, test)],
                 tf_ms, 0.0))

print(f"target: at most {MAX_ERROR:.0%} wrong among routed, backed by >= {MIN_ROUTED} DEV messages\n")
print(f"{'':10} {'threshold':>9} {'TEST routed':>12} {'wrong':>6} {'to a person':>12} "
      f"{'ms/msg':>7} {'$ per 1k':>9}")
table = []
for name, row_threshold, row_scores, row_correct, ms, cost in rows:
    if row_threshold is None:
        routed, wrong = 0, 0
        print(f"{name:10} {'none':>9} {0:>12} {0:>6} {len(test):>12} {ms:>7.0f} {cost:>9.3f}"
              "   no threshold meets the target")
    else:
        routed, wrong = price(row_threshold, row_scores, row_correct)
        print(f"{name:10} {row_threshold:>9.2f} {routed:>12} {wrong:>6} {len(test) - routed:>12} "
              f"{ms:>7.0f} {cost:>9.3f}")
    table.append((name, routed, wrong, ms, cost))
if any(row[0] == "zero-shot" for row in table):
    print("\nzero-shot is gated on its top probability alone: it has no out-of-scope question, "
          "while Jev's route_score includes the companion Noul.")

# The same table as a chart: one panel per measure, one bar per model.
_, axes = plt.subplots(1, 4, figsize=(13, 3.2))
for ax, (title, col, fmt) in zip(axes, [("TEST routed, of 150", 1, "{:.0f}"), ("wrong among routed", 2, "{:.0f}"),
                                        ("ms per message", 3, "{:.0f}"), ("$ per 1,000 messages", 4, "{:.3f}")]):
    values = [row[col] for row in table]
    ax.bar([row[0] for row in table], values, color=["#40B8A6", "#9CA3AF", "#1A3D4D"][:len(table)])
    ax.set_title(title, fontsize=10)
    ax.set_ylim(0, (max(values) or 1) * 1.25)
    for x, v in enumerate(values):
        ax.annotate(fmt.format(v), (x, v), ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()
# Explanation:
# - Ours: Jev picks 0.64 to 0.69 on DEV and routes 78 to 80 of 150 TEST messages with 0 wrong; the
#   zero-shot model meets the target at no threshold, so a person reads all 150.
# - Is that fair to the baseline? We measured the two obvious objections on DEV, in our planning runs: giving it the
#   intent names as labels (Jev's criteria) drops it to 36% right; adding an 11th label
#   "something else, none of these topics" catches 1 of 100 out-of-scope messages. And at equal
#   coverage, routing its 150 most confident DEV messages, 52 of them are wrong.
# - The trained TF-IDF row (BONUS DEMO) is Module 1's lesson again: with 15,000 labels it is
#   accurate (94%) and nearly free; its probabilities run low (said 0.39, right 0.67), so it
#   routes 37 of 150 at the target. Use it when you have labels for a fixed set of teams.
# - It has no API bill, but a classifier that can route nothing on its own costs you a person
#   per message. Compare on the metric your business pays for, not on milliseconds.
# - Common mistake: choosing the threshold on TEST. Then TEST is no longer a check.

**Figure: Why you pin versions - a threshold holds for the version it was tuned on, so a new release means a new eval before you switch.**

<!-- DIAGRAM: pin-versions -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/pin-versions.png" width="750" alt="Before the release: pin jev-1.13.0, log response.model, tune choose_threshold on DEV. Then jev-1.14.0 ships on 14 Oct 2026 and jev-latest answers with it, while your code did not change. Before you switch: re-run the DEV sweep and TEST check on jev-1.14.0; if the numbers held, set JEV_MODEL to jev-1.14.0; if not, retune first">

`ask_all()` keeps `response.model` with every answer, so each result says which version produced it.

In [ ]:
# ===========================================================================
# SOLUTION: LAB 3.3 (Challenge, optional)
# ===========================================================================
# Approach:
# - Same questions, same TEST messages, only the model id changes; the threshold stays fixed.
# Spec: version_check(old_model) needs jev_threshold from Lab 3.2. It asks the TEST messages
# again with old_model (same team_question_data and fits_question), applies jev_threshold, and prints
# routed and wrong for JEV_MODEL (from test_scores) and for old_model, then how many team answers
# changed between the two versions.
def version_check(old_model: str = "jev-1.13.0") -> None:
    if jev_threshold is None:
        print("Lab 3.2 chose no threshold, so there is nothing to compare.")
        return
    from typesafe_sdk import TypeSafeError
    try:
        old = ask_all(test, {"team": team_question_data, "fits": fits_question}, model=old_model)
    except TypeSafeError as error:        # an old model can be retired
        print(f"{old_model} did not answer ({type(error).__name__}): skip this check.")
        return
    old_scores = [route_score(r["answers"]) for r in old]
    old_correct = [label != "oos" and r["answers"]["team"].choice == label
                   for r, (_, label) in zip(old, test)]
    for name, scores, correct in ((JEV_MODEL, test_scores, test_correct),
                                  (old[0]["model"], old_scores, old_correct)):
        routed, wrong = price(jev_threshold, scores, correct)
        print(f"{name:11} threshold {jev_threshold:.2f}: routed {routed}, wrong {wrong} "
              f"({wrong / max(routed, 1):.1%})")
    changed = sum(a["answers"]["team"].choice != b["answers"]["team"].choice
                  for a, b in zip(test_answers, old))
    print(f"team answers that changed between versions: {changed} of {len(test)}")


version_check()
# Explanation:
# - Ours (two runs): 1.14 routed 78 to 80 with 0 wrong, 1.13 routed 73 to 77 with 0 wrong, and
#   4 to 8 of 150 team answers changed. The threshold happened to transfer here; you only know because you measured.
# - Before following jev-latest: re-run the DEV sweep and the TEST check with the new id, and
#   re-run the reliability table for every question your code gates on.

# Summary

## What you can now do

1. **Check a probability before you trust it.** Group the answers by what Jev said and count
   the right ones. With Module 1's few words per team, answers at 0.90 to 0.99 were right 75% to 78%
   of the time; with Module 2's intent lists, nearly all of them. Calibration belongs to the
   question as much as to the model.
2. **Catch what no team handles.** A Choice always picks; a companion Noul with the supported
   tasks as data caught 92 of 100 out-of-scope DEV messages in the same request.
3. **Choose the review line from data.** Say the target, sweep on DEV, check once on TEST, pin
   the model. At "2 wrong in 100" Jev routed about half of TEST on its own with none wrong.

**Figure: Module 3 in one picture (our run) - Jev, the zero-shot baseline and the optional trained classifier on accuracy, routing at a 2% target, speed and cost.**

<!-- DIAGRAM: summary-jev-vs-zero-shot -->
<img src="https://raw.githubusercontent.com/axel-sirota/getting_started_jev/main/course/diagrams/module_03/summary-jev-vs-zero-shot.png" width="750" alt="Three columns. Jev, jev-1.14.0: 81 percent of DEV teams right with a few words per team, 99 percent with intent lists; at a 2 percent target 80 of 150 TEST messages routed, 0 wrong; 62 ms per message, $0.09 per 1,000. Zero-shot, bart-large-mnli: 56 percent with a few words; 0 routed, all 150 to a person, and no out-of-scope question; 121 ms per message on CPU, no API bill. TF-IDF trained on 15,000 labels, optional: 94 percent; 37 of 150 routed, 0 wrong; under 1 ms, no API bill">

Like for like, with a few words per team, Jev got 81% and the zero-shot model 56%. The zero-shot
model has no out-of-scope question, and neither has TF-IDF: Jev's `route_score` includes the
companion Noul. In an extra run (not reproduced in this notebook), an 11th zero-shot label "none of
these" caught 1 of 100 out-of-scope messages. With 15,000 labels, train a classifier; Jev is for the judgments you have
no labels for.

### The honest part

The zero-shot model is the free, no-labels first try, not the strongest baseline: with a few
thousand labeled messages you could fine-tune a classifier, and you should measure that too
before you choose. Our sets are small: with 150 TEST messages, "0 wrong" means "probably under a
few percent". And some CLINC150 labels are arguable ("give me the weather forecast for today" is
labeled out of scope), so part of what we call an error is the label, not the model. In
production, keep labeling a sample of live traffic and re-run these checks on a schedule and on
every model bump; we measured p50 latency only, not the slow tail.

## Looking Ahead

Module 4, Jev and LLMs Together. Below our threshold a person reviews; above it we route to a
team. Next we add the third handler: an LLM that writes the reply, with Jev in front of it as a
router and behind it as a guardrail that checks every reply before it goes out.

## Homework

Take one Jev question from your own work (or Module 1's urgency Score). Label 50 examples by hand
(no real customer data), run the question, and build its reliability table. Where is it
overconfident? Write the target you would ship it with, in one sentence a manager would sign.